# ═══════════════════════════════════════════════════════════════
YOLO Training Notebook — Auto-generated by YOLO Label Studio
Model: yolov8s.pt  |  Task: detect  |  Epochs: 150
═══════════════════════════════════════════════════════════════
Run this notebook in Google Colab with GPU runtime enabled.
Runtime → Change runtime type → T4 GPU (or better)
═══════════════════════════════════════════════════════════════


## Cell 1: Check GPU & Install Dependencies


In [ ]:
import subprocess, sys

# Check GPU availability
gpu_check = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_check.returncode == 0:
    print("✅ GPU detected:")
    print(gpu_check.stdout[:500])
else:
    print("⚠️  No GPU detected! Training will be very slow.")
    print("   Go to Runtime → Change runtime type → T4 GPU")

# Install ultralytics (includes all YOLO versions)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'ultralytics>=8.3.0'])

# Verify installation
import ultralytics
ultralytics.checks()
print(f"\n✅ Ultralytics {ultralytics.__version__} installed successfully")


## Cell 2: Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_SAVE_PATH = '/content/drive/MyDrive/yolo_training'
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
print(f"✅ Drive mounted. Models will be saved to: {DRIVE_SAVE_PATH}")


## Cell 3: Upload & Extract Dataset


In [ ]:
import zipfile, shutil, glob
from pathlib import Path

# ─── OPTION A: Upload ZIP directly ───
from google.colab import files
print("📁 Upload your dataset ZIP (from YOLO Label Studio export):")
print("   Expected structure: dataset/images/ + dataset/labels/")
uploaded = files.upload()

# Extract the uploaded zip
WORK_DIR = Path('/content/yolo_project')
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

for fname in uploaded.keys():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall(WORK_DIR)
        print(f"✅ Extracted {fname} to {WORK_DIR}")

# ─── OPTION B (Alternative): Copy from Drive ───
# If you uploaded the ZIP to Drive instead, uncomment these lines:
# ZIP_PATH = DRIVE_SAVE_PATH + '/yolo_training_package.zip'
# with zipfile.ZipFile(ZIP_PATH, 'r') as z:
#     z.extractall(WORK_DIR)
# print(f"✅ Extracted from Drive: {ZIP_PATH}")

# Auto-detect the dataset structure
def find_folder(root, name):
    for p in Path(root).rglob(name):
        if p.is_dir():
            return p
    return None

img_dir = find_folder(WORK_DIR, 'images')
lbl_dir = find_folder(WORK_DIR, 'labels')

if img_dir is None or lbl_dir is None:
    # Flat structure fallback — images and labels at root
    all_imgs = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']:
        all_imgs.extend(list(WORK_DIR.rglob(ext)))
    if all_imgs:
        img_dir = all_imgs[0].parent
        lbl_dir = img_dir  # labels alongside images
        print(f"⚠️ Using flat structure from {img_dir}")
    else:
        raise FileNotFoundError("❌ Could not find images! Check your ZIP structure.")

print(f"📂 Images: {img_dir}")
print(f"📂 Labels: {lbl_dir}")

img_files = sorted([f for ext in ['*.jpg','*.jpeg','*.png','*.bmp','*.webp']
                     for f in img_dir.glob(ext)])
print(f"📊 Found {len(img_files)} images")


## Cell 4: Split Dataset into Train/Val


In [ ]:
import random

VAL_SPLIT = 0.1
random.seed(42)

# Create YOLO directory structure
DATASET_DIR = WORK_DIR / 'yolo_dataset'
for split in ['train', 'val']:
    (DATASET_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

# Shuffle and split
random.shuffle(img_files)
val_count = max(1, int(len(img_files) * VAL_SPLIT))
val_files = img_files[:val_count]
train_files = img_files[val_count:]

def copy_pair(img_path, split):
    """Copy image and its corresponding label to the split folder."""
    stem = img_path.stem
    ext = img_path.suffix

    # Copy image
    dst_img = DATASET_DIR / 'images' / split / img_path.name
    shutil.copy2(img_path, dst_img)

    # Copy label (look in labels dir, then same dir as image)
    label_name = stem + '.txt'
    label_src = lbl_dir / label_name
    if not label_src.exists():
        label_src = img_path.parent / label_name
    if label_src.exists():
        dst_lbl = DATASET_DIR / 'labels' / split / label_name
        shutil.copy2(label_src, dst_lbl)

for f in train_files:
    copy_pair(f, 'train')
for f in val_files:
    copy_pair(f, 'val')

print(f"✅ Dataset split complete:")
print(f"   Train: {len(train_files)} images")
print(f"   Val:   {len(val_files)} images")


## Cell 5: Create data.yaml


In [ ]:
import yaml

data_config = {
    'path': str(DATASET_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 3,
    'names': ['Circular', 'Rectangular', 'Triangular']
}

data_yaml_path = DATASET_DIR / 'data.yaml'
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

print("✅ data.yaml created:")
print(open(data_yaml_path).read())


## Cell 6: Verify Dataset (Visual Check)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np

CLASS_NAMES = ['Circular', 'Rectangular', 'Triangular']
COLORS = ['#6366f1','#22c55e','#ef4444','#f59e0b','#06b6d4',
          '#ec4899','#8b5cf6','#14b8a6','#f97316','#a3e635']

def show_sample(img_path, lbl_path, ax):
    img = Image.open(img_path)
    w, h = img.size
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')
    if lbl_path.exists():
        for line in open(lbl_path).read().strip().split('\n'):
            if not line.strip():
                continue
            parts = line.strip().split()
            cls_id = int(parts[0])
            cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            x1 = (cx - bw/2) * w
            y1 = (cy - bh/2) * h
            rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                                     linewidth=2, edgecolor=COLORS[cls_id % len(COLORS)],
                                     facecolor='none')
            ax.add_patch(rect)
            label = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
            ax.text(x1, y1-4, label, fontsize=7, color='white',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor=COLORS[cls_id % len(COLORS)], alpha=0.8))

# Show up to 6 sample images
train_imgs = sorted((DATASET_DIR / 'images' / 'train').glob('*'))
n_show = min(6, len(train_imgs))
if n_show > 0:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    for i in range(n_show):
        lbl_path = DATASET_DIR / 'labels' / 'train' / (train_imgs[i].stem + '.txt')
        show_sample(train_imgs[i], lbl_path, axes[i])
    for i in range(n_show, 6):
        axes[i].axis('off')
    plt.suptitle('Dataset Sample Verification', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print("✅ Verify that bounding boxes look correct before training!")
else:
    print("⚠️ No training images found to display")


## Cell 7: Train YOLO Model 🚀


In [ ]:
from ultralytics import YOLO

# Load pretrained model
MODEL_NAME = 'yolov8s.pt'
model = YOLO(MODEL_NAME)

print(f"🚀 Starting training with {MODEL_NAME}")
print(f"   Epochs: 150 | Batch: 32 | ImgSz: 640")
print(f"   LR: 0.001 | Patience: 20")
print(f"   Task: detect")
print("=" * 60)

# Train
results = model.train(
    data=str(data_yaml_path),
    epochs=150,
    batch=32,
    imgsz=640,
    lr0=0.001,
    patience=20,
    project=str(WORK_DIR / 'runs'),
    name='Hole_shape_detector ',
    exist_ok=True,
    plots=True,
    save=True,
    save_period=10,     # Save checkpoint every 10 epochs
    device=0,           # Use GPU 0
    workers=2,
    pretrained=True,
    verbose=True,
    val=True,
    amp=True,           # Mixed precision for faster training
)

print("\n" + "=" * 60)
print("✅ Training complete!")


## Cell 8: Evaluate & Show Results


In [ ]:
from IPython.display import Image as IPImage, display
import glob as g

run_dir = WORK_DIR / 'runs' / 'Hole_shape_detector '

# Display training curves
results_img = run_dir / 'results.png'
if results_img.exists():
    display(IPImage(filename=str(results_img), width=900))
    print("📊 Training curves shown above")

# Confusion matrix
cm_img = run_dir / 'confusion_matrix.png'
if cm_img.exists():
    display(IPImage(filename=str(cm_img), width=700))
    print("📊 Confusion matrix shown above")

# Validation predictions
val_pred = run_dir / 'val_batch0_pred.jpg'
if val_pred.exists():
    display(IPImage(filename=str(val_pred), width=900))
    print("📊 Validation predictions shown above")

# Print final metrics
metrics = model.val()
print("\n📊 Final Validation Metrics:")
print(f"   mAP50:     {metrics.box.map50:.4f}")
print(f"   mAP50-95:  {metrics.box.map:.4f}")


## Cell 9: Save Model to Google Drive 💾


In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir = Path(DRIVE_SAVE_PATH) / f'Hole_shape_detector _{timestamp}'
save_dir.mkdir(parents=True, exist_ok=True)

# Copy best weights
best_pt = run_dir / 'weights' / 'best.pt'
last_pt = run_dir / 'weights' / 'last.pt'

if best_pt.exists():
    shutil.copy2(best_pt, save_dir / 'best.pt')
    print(f"✅ best.pt saved to {save_dir / 'best.pt'}")
else:
    print("⚠️ best.pt not found")

if last_pt.exists():
    shutil.copy2(last_pt, save_dir / 'last.pt')
    print(f"✅ last.pt saved to {save_dir / 'last.pt'}")

# Copy training results and plots
for f in run_dir.glob('*.png'):
    shutil.copy2(f, save_dir / f.name)
for f in run_dir.glob('*.csv'):
    shutil.copy2(f, save_dir / f.name)

# Copy the data.yaml for reference
shutil.copy2(data_yaml_path, save_dir / 'data.yaml')

# Also save a convenience copy of best.pt at root of drive path
shutil.copy2(best_pt, Path(DRIVE_SAVE_PATH) / 'best.pt')
print(f"\n💾 Convenience copy: {Path(DRIVE_SAVE_PATH) / 'best.pt'}")

print(f"\n{'=' * 60}")
print(f"🎉 All files saved to Google Drive!")
print(f"   📂 {save_dir}")
print(f"\n🤖 For your drone application, use:")
print(f"   model = YOLO('{Path(DRIVE_SAVE_PATH) / 'best.pt'}')")
print(f"   results = model.predict(source=frame, conf=0.5)")


## Cell 10: Export to Other Formats (Optional)


In [ ]:
# Uncomment the export format you need for your drone hardware:

# Export to ONNX (widely supported, good for most edge devices)
# model.export(format='onnx', imgsz=640, simplify=True)

# Export to TensorRT (NVIDIA Jetson / GPU edge devices)
# model.export(format='engine', imgsz=640, half=True)

# Export to TFLite (Raspberry Pi, Android, lightweight edge)
# model.export(format='tflite', imgsz=640)

# Export to CoreML (Apple devices / iOS)
# model.export(format='coreml', imgsz=640)

# Export to NCNN (lightweight, C++ based, good for ARM/mobile)
# model.export(format='ncnn', imgsz=640)

# After exporting, copy to Drive:
# import shutil
# for f in run_dir.glob('weights/*'):
#     shutil.copy2(f, save_dir / f.name)
# print("Exported models copied to Drive!")

print("\n✅ Training pipeline complete!")

